# Insighta NLP Training (DistilBERT, Colab GPU)

This notebook trains a two-head DistilBERT model for `categoryName` and `priority`, then applies a hybrid priority decision (`ML + rules`) for inference.

Artifacts exported for backend use:
- `model/`
- `label_maps_categoryName.json`
- `label_maps_priority.json`
- `temperature_scaling.json`
- `nlp_priority_rules.json`
- `inference_config.json`
- `training_summary.json`


## 1) Setup


In [ ]:
# Colab setup
!pip -q install -U transformers datasets evaluate scikit-learn accelerate


In [ ]:
import json
import random
import shutil
from dataclasses import dataclass
from datetime import datetime
from pathlib import Path
from typing import Dict, List

import numpy as np
import torch
import torch.nn as nn

from datasets import Dataset
from google.colab import drive
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

from transformers import (
    AutoConfig,
    AutoTokenizer,
    DistilBertModel,
    DistilBertPreTrainedModel,
    Trainer,
    TrainingArguments,
    set_seed,
)

print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
# Reproducibility
SEED = 42
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


## 2) Paths and Data Loading


In [ ]:
# Mount Drive and configure paths
drive.mount('/content/drive')

# Update BASE_DIR to where your repo/data lives in Drive
BASE_DIR = Path('/content/drive/MyDrive/Insighta')
DATA_DIR = BASE_DIR / 'data' / 'nlp'

RUN_ID = datetime.utcnow().strftime('%Y%m%d_%H%M%S')
OUTPUT_DIR = BASE_DIR / 'artifacts' / f'distilbert_complaint_twohead_{RUN_ID}'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = DATA_DIR / 'train.jsonl'
VAL_PATH = DATA_DIR / 'val.jsonl'
TEST_PATH = DATA_DIR / 'test.jsonl'

for p in [TRAIN_PATH, VAL_PATH, TEST_PATH]:
    if not p.exists():
        raise FileNotFoundError(f'Missing dataset file: {p}')

print('RUN_ID:', RUN_ID)
print('Data dir:', DATA_DIR)
print('Output dir:', OUTPUT_DIR)


In [ ]:
# Load JSONL rows
def load_jsonl(path: Path) -> List[Dict]:
    rows = []
    with path.open('r', encoding='utf-8') as f:
        for i, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue
            rows.append(json.loads(line))
    return rows

train_rows = load_jsonl(TRAIN_PATH)
val_rows = load_jsonl(VAL_PATH)
test_rows = load_jsonl(TEST_PATH)

print('train:', len(train_rows), 'val:', len(val_rows), 'test:', len(test_rows))


## 3) Labels and Encoding


In [ ]:
# Label vocab (taxonomy-locked)
CATEGORY_LABELS = [
    'Policy & Account Servicing',
    'Claims Experience',
    'Payments, Billing & Refunds',
    'Documents & Requirements',
    'Customer Support & Service Quality',
    'Digital Access & Technical Issues',
    'Fraud, Security & Privacy',
    'Product/Partner Service Delivery',
    'Other / Uncategorized',
]
PRIORITY_LABELS = ['Low', 'Med', 'High']

category2id = {v: i for i, v in enumerate(CATEGORY_LABELS)}
priority2id = {v: i for i, v in enumerate(PRIORITY_LABELS)}


def encode_rows(rows: List[Dict]) -> List[Dict]:
    encoded = []
    for row in rows:
        labels = row['labels']
        c = labels['categoryName']
        p = labels['priority']

        if c not in category2id:
            raise ValueError(f'Unknown categoryName: {c}')
        if p not in priority2id:
            raise ValueError(f'Unknown priority: {p}')

        encoded.append({
            'text': row['text'],
            'labels_category': category2id[c],
            'labels_priority': priority2id[p],
        })
    return encoded

train_enc = encode_rows(train_rows)
val_enc = encode_rows(val_rows)
test_enc = encode_rows(test_rows)

print('Encoded rows ready.')
print('Sample:', train_enc[0])


In [ ]:
# Save label maps for backend integration
label_maps = {
    'categoryName': {'id2label': {str(i): v for i, v in enumerate(CATEGORY_LABELS)}, 'label2id': category2id},
    'priority': {'id2label': {str(i): v for i, v in enumerate(PRIORITY_LABELS)}, 'label2id': priority2id},
}
for name, mapping in label_maps.items():
    (OUTPUT_DIR / f'label_maps_{name}.json').write_text(json.dumps(mapping, indent=2), encoding='utf-8')
print('Wrote label maps to', OUTPUT_DIR)


## 4) Tokenization and Class Balance Check


In [ ]:
# Build datasets + tokenizer
train_ds = Dataset.from_list(train_enc)
val_ds = Dataset.from_list(val_enc)
test_ds = Dataset.from_list(test_enc)

MODEL_NAME = 'distilbert-base-uncased'
MAX_LENGTH = 320

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_batch(batch):
    return tokenizer(batch['text'], truncation=True, max_length=MAX_LENGTH)

train_ds = train_ds.map(tokenize_batch, batched=True)
val_ds = val_ds.map(tokenize_batch, batched=True)
test_ds = test_ds.map(tokenize_batch, batched=True)

print(train_ds)


In [ ]:
# Quick class-imbalance check (max/min count ratio)


def class_ratio_report(name: str, labels: np.ndarray, label_names: List[str], acceptable_max_ratio: float = 1.3):
    counts = np.bincount(labels, minlength=len(label_names)).astype(np.int64)
    min_count = int(counts.min())
    max_count = int(counts.max())
    ratio = float(max_count / max(min_count, 1))

    print(f'\n{name} distribution:')
    for i, c in enumerate(counts):
        print(f'  {i:>2} | {label_names[i]}: {int(c)}')

    print(f'{name} max/min ratio: {ratio:.3f}')
    if ratio <= acceptable_max_ratio:
        print(f'{name} balance check PASSED (<= {acceptable_max_ratio}).')
    else:
        print(f'{name} balance check WARNING (> {acceptable_max_ratio}).')

    return {'counts': counts.tolist(), 'max_min_ratio': ratio}


category_train = np.array([r['labels_category'] for r in train_enc], dtype=np.int64)
priority_train = np.array([r['labels_priority'] for r in train_enc], dtype=np.int64)

balance_report = {
    'categoryName': class_ratio_report('categoryName', category_train, CATEGORY_LABELS, acceptable_max_ratio=1.3),
    'priority': class_ratio_report('priority', priority_train, PRIORITY_LABELS, acceptable_max_ratio=1.3),
}


## 5) Model, Metrics, and Trainer


In [ ]:
# Data collator
@dataclass
class MultiTaskDataCollator:
    tokenizer: object

    def __call__(self, features):
        label_keys = ['labels_category', 'labels_priority']
        labels = {k: [f[k] for f in features] for k in label_keys}
        clean_features = []
        for f in features:
            g = dict(f)
            for k in label_keys:
                g.pop(k, None)
            # keep only model fields
            g = {k: v for k, v in g.items() if k in {'input_ids', 'attention_mask', 'token_type_ids'}}
            clean_features.append(g)

        batch = self.tokenizer.pad(clean_features, return_tensors='pt')
        batch['labels_category'] = torch.tensor(labels['labels_category'], dtype=torch.long)
        batch['labels_priority'] = torch.tensor(labels['labels_priority'], dtype=torch.long)
        return batch

collator = MultiTaskDataCollator(tokenizer=tokenizer)


In [ ]:
# Two-head DistilBERT model (categoryName + priority)
class MultiTaskDistilBert(DistilBertPreTrainedModel):
    def __init__(self, config):
        super().__init__(config)
        self.num_labels_category = config.num_labels_category
        self.num_labels_priority = config.num_labels_priority

        self.distilbert = DistilBertModel(config)
        self.dropout = nn.Dropout(config.seq_classif_dropout)

        hidden = config.dim
        self.classifier_category = nn.Linear(hidden, self.num_labels_category)
        self.classifier_priority = nn.Linear(hidden, self.num_labels_priority)

        self.post_init()

    def forward(
        self,
        input_ids=None,
        attention_mask=None,
        labels_category=None,
        labels_priority=None,
        **kwargs,
    ):
        outputs = self.distilbert(input_ids=input_ids, attention_mask=attention_mask)
        hidden_state = outputs.last_hidden_state
        pooled = hidden_state[:, 0]
        pooled = self.dropout(pooled)

        logits_category = self.classifier_category(pooled)
        logits_priority = self.classifier_priority(pooled)

        loss = None
        if all(x is not None for x in [labels_category, labels_priority]):
            ce = nn.CrossEntropyLoss()
            l_category = ce(logits_category, labels_category)
            l_priority = ce(logits_priority, labels_priority)
            loss = l_category + l_priority

        return {
            'loss': loss,
            'logits': (logits_category, logits_priority),
        }

config = AutoConfig.from_pretrained(MODEL_NAME)
config.num_labels_category = len(CATEGORY_LABELS)
config.num_labels_priority = len(PRIORITY_LABELS)

model = MultiTaskDistilBert.from_pretrained(MODEL_NAME, config=config)
print('Model initialized')


In [ ]:
# Metrics for Trainer

def compute_metrics(eval_pred):
    preds = eval_pred.predictions
    labels = eval_pred.label_ids

    if not isinstance(preds, (tuple, list)) or len(preds) != 2:
        raise ValueError('Expected 2 prediction tensors for multi-task outputs.')
    if not isinstance(labels, (tuple, list)) or len(labels) != 2:
        raise ValueError('Expected 2 label tensors for multi-task outputs.')

    y_cat, y_prio = labels
    p_cat = np.argmax(preds[0], axis=-1)
    p_prio = np.argmax(preds[1], axis=-1)

    metrics = {
        'category_acc': accuracy_score(y_cat, p_cat),
        'priority_acc': accuracy_score(y_prio, p_prio),
        'category_macro_f1': f1_score(y_cat, p_cat, average='macro'),
        'priority_macro_f1': f1_score(y_prio, p_prio, average='macro'),
    }
    metrics['macro_f1_avg'] = float(np.mean([
        metrics['category_macro_f1'],
        metrics['priority_macro_f1'],
    ]))
    return metrics


In [ ]:
# Training config
training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR / 'checkpoints'),
    num_train_epochs=6,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=1e-5,
    weight_decay=0.01,
    logging_steps=25,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='macro_f1_avg',
    greater_is_better=True,
    save_total_limit=2,
    fp16=torch.cuda.is_available(),
    report_to='none',
    seed=SEED,
    label_names=['labels_category', 'labels_priority'],
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=collator,
    compute_metrics=compute_metrics,
)

print('Trainer ready')


## 6) Train and Evaluate


In [ ]:
# Train
train_result = trainer.train()
print(train_result)


In [ ]:
# Evaluate
val_pred = trainer.predict(val_ds)
val_metrics = compute_metrics(val_pred)

print("Validation metrics:")
print(json.dumps(val_metrics, indent=2))

test_pred = trainer.predict(test_ds)
test_metrics = compute_metrics(test_pred)

print("Test metrics:")
print(json.dumps(test_metrics, indent=2))


## 7) Temperature Scaling Calibration


In [ ]:
# Temperature scaling calibration (per head) using the validation set
import torch
import torch.nn.functional as F

def fit_temperature(logits: torch.Tensor, labels: torch.Tensor, iters: int = 800, lr: float = 0.05):
    """
    Fits scalar temperature T by minimizing NLL on validation logits.
    Returns a float T.
    """
    T = torch.nn.Parameter(torch.ones(1, device=logits.device))
    opt = torch.optim.Adam([T], lr=lr)

    for _ in range(iters):
        opt.zero_grad()
        loss = F.cross_entropy(logits / T.clamp_min(1e-6), labels)
        loss.backward()
        opt.step()

    return float(T.detach().cpu().item())

# 1) Get validation logits + labels from the Trainer
val_pred = trainer.predict(val_ds)
logits_cat_np, logits_prio_np = val_pred.predictions
y_cat_np, y_prio_np = val_pred.label_ids

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

logits_cat = torch.tensor(logits_cat_np, dtype=torch.float32, device=device)
logits_prio = torch.tensor(logits_prio_np, dtype=torch.float32, device=device)
y_cat = torch.tensor(y_cat_np, dtype=torch.long, device=device)
y_prio = torch.tensor(y_prio_np, dtype=torch.long, device=device)

# 2) Fit temperatures
T_CAT = fit_temperature(logits_cat, y_cat)
T_PRIO = fit_temperature(logits_prio, y_prio)

print("✅ Temperature scaling fitted:")
print("T_CAT =", T_CAT)
print("T_PRIO =", T_PRIO)

## 8) Reports


In [ ]:
# Detailed reports, hard-case metrics, confusion matrices
preds = test_pred.predictions
labels = test_pred.label_ids
if not (isinstance(preds, (tuple, list)) and isinstance(labels, (tuple, list)) and len(preds) == 2 and len(labels) == 2):
    raise ValueError('Unexpected test prediction format.')

y_cat, y_prio = labels

p_cat = np.argmax(preds[0], axis=-1)
p_prio = np.argmax(preds[1], axis=-1)

reports = {
    'categoryName': classification_report(y_cat, p_cat, target_names=CATEGORY_LABELS, output_dict=True),
    'priority': classification_report(y_prio, p_prio, target_names=PRIORITY_LABELS, output_dict=True),
}

confusions = {
    'categoryName': confusion_matrix(y_cat, p_cat).tolist(),
    'priority': confusion_matrix(y_prio, p_prio).tolist(),
}

(OUTPUT_DIR / 'classification_reports.json').write_text(json.dumps(reports, indent=2), encoding='utf-8')
(OUTPUT_DIR / 'confusion_matrices.json').write_text(json.dumps(confusions, indent=2), encoding='utf-8')

print('Saved classification reports, confusion matrices, and hard-case metrics.')


## 9) Artifact Export (Model + Rule Config)


In [ ]:
# Save model and summary
MODEL_OUT = OUTPUT_DIR / 'model'
MODEL_OUT.mkdir(parents=True, exist_ok=True)

trainer.model.save_pretrained(MODEL_OUT, safe_serialization=True)
tokenizer.save_pretrained(MODEL_OUT)

priority_rules = {
    'category_base': {
        'Fraud, Security & Privacy': 'High',
        'Claims Experience': 'High',
        'Payments, Billing & Refunds': 'High',

        'Digital Access & Technical Issues': 'Med',
        'Policy & Account Servicing': 'Med',
        'Product/Partner Service Delivery': 'Med',

        'Documents & Requirements': 'Low',
        'Customer Support & Service Quality': 'Low',
        'Other / Uncategorized': 'Low',
    },

    # ── STRONG HIGH ─────────────────────────────────────
    # Should push to High on their own. These indicate
    # real financial harm, security risk, safety, or
    # escalation intent.
    'high_strong_patterns': [

        # ── Urgency / deadlines (with context) ──
        r'\burgent\b',
        r'\basap\b',
        r'\bimmediately\b',
        r'\bright away\b',
        r'\bright now\b',
        r'\btime.?sensitive\b',
        r'\bdeadline\b',
        r'\bdue date\b',
        r'\b(by|before|until)\s+(today|tomorrow|end of day|eod|tonight)\b',
        r'\b(already|is|due|due\s+on|this\s+coming)\s+(today|tomorrow|tonight)\b',
        r'\b(tomorrow|today)\b.*\b(deadline|due|pay|payment|balance|lapse|expir)\b',
        r'\b(pay|payment|balance)\b.*\b(tomorrow|today|tonight)\b',
        r'within\s*\d+\s*(hours?|hrs?|days?|business days?)\b',

        # ── Escalation / legal / regulatory ──
        r'\bescalat(e|ed|ion|ing)\b',
        r'\bformal\s+complaint\b',
        r'\bfile\s+(a\s+)?complaint\b',
        r'\blegal\s*(action|counsel|proceedings?)?\b',
        r'\blawyer\b',
        r'\battorney\b',
        r'\bpolice\b',
        r'\bmedia\b',
        r'\bsocial media\b',
        r'\bDTI\b',
        r'\bInsurance Commission\b',
        r'\bBSP\b',
        r'\bregulator\b',
        r'\bombudsman\b',
        r'\bconsumer\s+(protection|rights)\b',

        # ── Fraud / security / account compromise ──
        r'\bfraud(ulent)?\b',
        r'\bscam(med)?\b',
        r'\bhack(ed|ing)?\b',
        r'\bunauthori[sz]ed\b',
        r'\bbreach(ed)?\b',
        r'\bidentity\s*theft\b',
        r'\bphish(ing|ed)?\b',
        r'\bstolen\b',
        r'\bcompromised\b',
        r'\bsuspicious\s+(activity|transaction|charge|login)\b',
        r'\bOTP\b.*\b(share|gave|sent|someone)\b',
        r'\bsomeone\b.*\b(access|log|account)\b',

        # ── Direct financial harm ──
        r'\bcharged\s+(twice|2x|two\s+times|the\s+wrong|incorrectly)\b',
        r'\bdouble\s+charge[d]?\b',
        r'\bovercharg(e[d]?|ing)\b',
        r'\bwrong\s+(amount|charge|fee|deduction|bill)\b',
        r'\bincorrect\s+(amount|charge|fee|deduction|bill)\b',
        r'\b(extra|unexpected|unknown)\s+(charge|fee|deduction)\b',
        r'\bunauthori[sz]ed\s+(charge|deduction|transaction|payment|debit)\b',
        r'\bmoney\b.*\b(gone|missing|disappeared|deducted|taken)\b',
        r'\b(where|what happened to)\b.*\b(refund|money|payment|funds)\b',
        r'\brefund\b.*\b(not|never|still|haven.?t)\b',
        r'\b(not|never|still|haven.?t)\b.*\brefund\b',
        r'\bpayment\s+(failed|declined|rejected|not going through)\b',
        r'\b(lost|missing)\s+(payment|funds|money|deposit)\b',

        # ── Health / safety / emergency (insurance domain) ──
        r'\bhospitali[sz](ed|ation)\b',
        r'\bemergency\b',
        r'\baccident\b',
        r'\bdeath\b',
        r'\bdeceased\b',
        r'\bcritical\s*(illness|condition)?\b',
        r'\blife.?threatening\b',
        r'\bambulance\b',
        r'\bICU\b',
        r'\bsurgery\b',

        # ── Churn / cancellation threats ──
        r'\b(cancel|terminat)(e|ing|ed)\s*(my|the|this)?\s*(policy|account|plan|subscription)\b',
        r'\bswitch(ing)?\s*(to\s+)?(another|different|competitor)\b',
        r'\bclose\s+my\s+account\b',
        r'\bpull(ing)?\s+out\b',
    ],

    # ── WEAK HIGH ───────────────────────────────────────
    # Signals a blocker or access issue. Alone → Medium.
    # Combined with deadline/repeat/strong → can push High.
    'high_weak_patterns': [

        # ── Login / access blockers ──
        r'\b(can.?t|cannot|unable\s+to)\s*(log\s*in|sign\s*in|access)\b',
        r'\b(fix|broken|issue\s+with)\s*(the\s+)?(log\s*in|sign\s*in|login|access)\b',
        r'\blog\s*in\s*(issue|problem|broken|not\s+working|fail)\b',
        r'\blocked\s*(out|account)\b',
        r'\baccount\s*(disabled|locked|suspended|blocked|restricted)\b',
        r'\bsession\s*(expired|timed?\s*out)\b',

        # ── App / portal failures ──
        r'\b(portal|app|website|site|page|system)\s*(down|crash|not\s+working|unavailable)\b',
        r'\bcrash(es|ed|ing)?\b',
        r'\b(keeps?\s+)?(crash|freez|hang)(es|ed|ing)\b',
        r'\bnot\s+(loading|responding|opening|displaying)\b',
        r'\bblank\s+(screen|page)\b',
        r'\bwhite\s+screen\b',
        r'\b(500|502|503|504)\s*(error)?\b',
        r'\bserver\s+error\b',
        r'\btimeout\b',
        r'\btimed?\s*out\b',

        # ── Functional blockers ──
        r'\bcan.?t\s+(proceed|continue|submit|complete|upload|download|pay)\b',
        r'\bunable\s+to\s+(proceed|continue|submit|complete|upload|download|pay)\b',
        r'\b(button|link|form)\s*(not|won.?t)\s*(work|click|submit|load)\b',
        r'\bpayment\s*(page|gateway|portal)\b.*\b(error|not|won.?t|fail)\b',
        r'\bOTP\s*(not|never|didn.?t)\s*(received?|arrived?|come|sent)\b',

        # ── General error/failure (need context to be high) ──
        r'\berror\s*(code|message)?\s*[:=]?\s*\w+\b',
        r'\bfailed\b',
        r'\bsystem\s+error\b',
    ],

    # ── MEDIUM ──────────────────────────────────────────
    # Repeated issues, waiting, follow-ups, moderate
    # frustration. Not an emergency but needs attention.
    'med_patterns': [

        # ── Repetition / unresolved ──
        r'\b(again|yet again)\b',
        r'\bstill\s+(not|unresolved|pending|waiting|the same|broken)\b',
        r'\b(not|never)\s+(fixed|resolved|addressed|handled)\b',
        r'\bunresolved\b',
        r'\bsame\s+(issue|problem|error)\b',
        r'\b(second|third|fourth|\d+\s*(th|nd|rd|st))\s+time\b',
        r'\b(multiple|several|many|numerous)\s+times\b',
        r'\breported\b.*\b(times|multiple|several|already|before)\b',
        r'\balready\s+(reported|called|emailed|contacted|submitted|raised)\b',

        # ── Waiting / delays ──
        r'\bwaiting\b',
        r'\bpending\b',
        r'\bdelayed?\b',
        r'\bslow\b',
        r'\b(no|without)\s+(response|reply|callback|update|feedback|acknowledgment)\b',
        r'\bbeen\s+\d+\s*(days?|weeks?|months?)\b',
        r'\b(it.?s|its)\s+been\s+(days|weeks|a\s+while|so\s+long|forever)\b',
        r'\bhow\s+long\b.*\b(take|wait|more)\b',
        r'\bwhen\s+will\b',

        # ── Follow-up / request for attention ──
        r'\bfollow[- ]?up\b',
        r'\bstatus\s*(update|check)?\b',
        r'\bplease\s+(check|look into|assist|help|resolve|address|prioriti[sz]e)\b',
        r'\bkindly\b',
        r'\breview\b',
        r'\bticket\b',
        r'\breference\s*(number|#|no\.?|num)?\b',
        r'\bcase\s*(number|#|no\.?|num|id)?\b',

        # ── Moderate dissatisfaction ──
        r'\bdisappoint(ed|ing|ment)\b',
        r'\bfrustrat(ed|ing|ion)\b',
        r'\binconvenien(ce|t|ced)\b',
        r'\bunacceptable\b',
        r'\bridiculous\b',
        r'\b(poor|terrible|horrible|worst)\s+(service|experience|support)\b',
        r'\bnot\s+(happy|satisfied|pleased)\b',

        # ── Promised but not delivered ──
        r'\bpromised\b',
        r'\bassured\b',
        r'\b(told|said)\s+(me|us|it\s+would)\b',
        r'\bsupposed\s+to\b',
        r'\bexpect(ed|ing)?\b.*\b(but|however|still|yet)\b',
    ],

    # ── LOW ─────────────────────────────────────────────
    # Pure information-seeking. No urgency, harm, or
    # frustration signals.
    'low_patterns': [

        # ── Inquiry / question ──
        r'\binquir(y|e|ing|ies)\b',
        r'\bquestion\b',
        r'\bjust\s+(asking|wondering|curious|checking|inquiring)\b',
        r'\bcan\s+you\s+explain\b',
        r'\bclarif(y|ication)\b',
        r'\bwhat\s+(is|are|does)\b',
        r'\bhow\s+(to|do|does|can|much|many)\b',
        r'\bwhere\s+(can|do|is)\b',
        r'\bis\s+(it|there|this)\b.*\b(possible|available|covered)\b',

        # ── Informational / browsing ──
        r'\binformation\b',
        r'\bdetails\b.*\b(about|on|regarding)\b',
        r'\bguidelines?\b',
        r'\brequirements?\b',
        r'\bprocess\b.*\b(for|of|to)\b',
        r'\bprocedure\b',
        r'\bcoverage\b',
        r'\bbenefits?\b',
        r'\binclusions?\b',
        r'\beligib(le|ility)\b',

        # ── Routine account maintenance ──
        r'\b(update|change|edit)\s+(my\s+)?(address|email|number|name|details|info)\b',
        r'\bcopy\s+of\b',
        r'\brequest\s+(for|a)\s+(copy|document|certificate|form)\b',
        r'\brenewal\b.*\b(date|when|how)\b',
        r'\bhow\s+much\b',
        r'\bschedule\b',
    ],

    'ml_priority_conf_threshold': 0.75,
}

inference_config = {
    'max_length': MAX_LENGTH,
    'model_name': MODEL_NAME,
    'label_normalization': {'Med': 'Medium'},
    'outputs': ['categoryName', 'priority', 'confidenceCategory', 'confidencePriority', 'prioritySource'],
}

(OUTPUT_DIR / 'nlp_priority_rules.json').write_text(json.dumps(priority_rules, indent=2), encoding='utf-8')
(OUTPUT_DIR / 'inference_config.json').write_text(json.dumps(inference_config, indent=2), encoding='utf-8')

summary = {
    'run_id': RUN_ID,
    'timestamp_utc': datetime.utcnow().isoformat() + 'Z',
    'model_name': MODEL_NAME,
    'seed': SEED,
    'max_length': MAX_LENGTH,
    'train_size': len(train_ds),
    'val_size': len(val_ds),
    'test_size': len(test_ds),
    'balance_report': {
        'categoryName': {
            'max_min_ratio': balance_report['categoryName']['max_min_ratio'],
            'counts': balance_report['categoryName']['counts'],
        },
        'priority': {
            'max_min_ratio': balance_report['priority']['max_min_ratio'],
            'counts': balance_report['priority']['counts'],
        },
    },
    'training_args': {
        'epochs': training_args.num_train_epochs,
        'train_batch_size': training_args.per_device_train_batch_size,
        'eval_batch_size': training_args.per_device_eval_batch_size,
        'learning_rate': training_args.learning_rate,
        'weight_decay': training_args.weight_decay,
    },
    'temperature_scaling': {'T_CAT': T_CAT, 'T_PRIO': T_PRIO},
    'priority_rules': {'ml_priority_conf_threshold': 0.75},
    'val_metrics': val_metrics,
    'test_metrics': test_metrics,
}

(OUTPUT_DIR / 'training_summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')

zip_base = str(OUTPUT_DIR / f'insighta_distilbert_model_only_{RUN_ID}')
zip_path = shutil.make_archive(zip_base, 'zip', root_dir=MODEL_OUT)
print('Saved model to:', MODEL_OUT)
print('Wrote summary to:', OUTPUT_DIR / 'training_summary.json')
print('Wrote rules to:', OUTPUT_DIR / 'nlp_priority_rules.json')
print('Wrote inference config to:', OUTPUT_DIR / 'inference_config.json')
print('Model-only zip artifact:', zip_path)


## 10) Hybrid Inference Demo (ML + Rules)


In [ ]:
# Inference demo (hybrid: ML + keyword rules) — STRONG vs WEAK high + safeguard
import re
import numpy as np
import torch
from typing import List

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = trainer.model.to(DEVICE)
model.eval()

# --------- Load rule config ----------
CATEGORY_BASE = priority_rules['category_base']
MED_PATTERNS  = priority_rules['med_patterns']
LOW_PATTERNS  = priority_rules['low_patterns']
ML_PRIORITY_CONF_THRESHOLD = float(priority_rules.get('ml_priority_conf_threshold', 0.75))

def softmax_1d(x):
    x = x - np.max(x)
    ex = np.exp(x)
    return ex / ex.sum()

def _count_matches(text: str, patterns):
    return sum(1 for p in patterns if re.search(p, text, flags=re.IGNORECASE))

def _matched_patterns(text: str, patterns):
    hits = []
    for p in patterns:
        if re.search(p, text, flags=re.IGNORECASE):
            hits.append(p)
    return hits

def _norm_priority(v: str) -> str:
    return 'Medium' if v == 'Med' else v

# --------- Build strong vs weak high patterns ----------
# Preferred: user supplies these lists in priority_rules.
# Fallback: split old high_patterns using simple heuristics.
if "high_strong_patterns" in priority_rules and "high_weak_patterns" in priority_rules:
    HIGH_STRONG_PATTERNS = priority_rules["high_strong_patterns"]
    HIGH_WEAK_PATTERNS   = priority_rules["high_weak_patterns"]
else:
    # Fallback split from old list
    HIGH_PATTERNS = priority_rules["high_patterns"]

    # Anything legal/fraud/security/deadline/escalation-like => strong
    strong_markers = [
        "fraud", "scam", "hacked", "unauthorized", "breach", "identity",
        "phishing", "stolen", "otp", "compromised",
        "urgent", "asap", "immediately", "deadline", "within",
        "escalat", "legal", "lawyer", "police", "media", "dti", "insurance commission", r"\bic\b"
    ]
    HIGH_STRONG_PATTERNS, HIGH_WEAK_PATTERNS = [], []
    for p in HIGH_PATTERNS:
        pl = p.lower()
        if any(m in pl for m in strong_markers):
            HIGH_STRONG_PATTERNS.append(p)
        else:
            HIGH_WEAK_PATTERNS.append(p)

# --------- Rule-based priority ----------
def rule_priority(text: str, category: str):
    base = _norm_priority(CATEGORY_BASE.get(category, 'Low'))

    # Match patterns
    hi_strong_hits = _matched_patterns(text, HIGH_STRONG_PATTERNS)
    hi_weak_hits   = _matched_patterns(text, HIGH_WEAK_PATTERNS)
    med_hits       = _matched_patterns(text, MED_PATTERNS)
    low_hits       = _matched_patterns(text, LOW_PATTERNS)

    hi_strong = len(hi_strong_hits)
    hi_weak   = len(hi_weak_hits)
    med       = len(med_hits)
    low       = len(low_hits)

    # Score with caps to avoid pattern explosion
    base_score = {'Low': 0, 'Medium': 1, 'High': 2}[base]
    score = base_score \
        + 2 * min(hi_strong, 2) \
        + 1 * min(hi_weak, 2) \
        + 1 * min(med, 2) \
        - 1 * min(low, 1)

    # Decision:
    # High requires at least one STRONG high indicator,
    # OR base High with strong combined evidence.
    if (hi_strong >= 1 and score >= 3) or (base == "High" and (hi_strong + med) >= 2 and score >= 3):
        pr = 'High'
    elif score >= 1:
        pr = 'Medium'
    else:
        pr = 'Low'

    # Confidence from evidence strength (bounded)
    evidence = max(2 * min(hi_strong, 2) + 1 * min(hi_weak, 2) + 1 * min(med, 2) - 1 * min(low, 1), 0)
    conf = min(0.50 + 0.10 * evidence, 0.99)

    # If base is High, never output Low
    if base == "High" and pr == "Low":
        pr = "Medium"
        conf = max(conf, 0.55)  # optional: raise a bit since base says it's at least Medium

    # --------- Fix #4 safeguard ----------
    # If rule outputs High but there is no STRONG high signal, downgrade to Medium
    if pr == "High" and hi_strong == 0:
        pr = "Medium"
        conf = min(conf, 0.75)

    return pr, float(conf), {
        'base': base,
        'hi_strong': hi_strong,
        'hi_weak': hi_weak,
        'med': med,
        'low': low,
        'score': score,
        'hits': {
            'hi_strong': hi_strong_hits,
            'hi_weak': hi_weak_hits,
            'med': med_hits,
            'low': low_hits,
        }
    }

def top2_margin(probs: np.ndarray) -> float:
    s = np.sort(probs)
    return float(s[-1] - s[-2]) if len(s) >= 2 else 0.0


# --------- Predict texts (hybrid ML + rule) ----------
def predict_texts(texts: List[str]):
    batch = tokenizer(
        texts,
        truncation=True,
        max_length=MAX_LENGTH,
        padding=True,
        return_tensors='pt'
    )
    batch = {k: v.to(DEVICE) for k, v in batch.items()}

    with torch.no_grad():
        out = model(**batch)

    logits_cat, logits_prio = [x.cpu().numpy() for x in out['logits']]

    rows = []
    for i, text in enumerate(texts):
        pc = softmax_1d(logits_cat[i] / T_CAT)
        pp = softmax_1d(logits_prio[i] / T_PRIO)

        pred_category = CATEGORY_LABELS[int(np.argmax(pc))]
        ml_priority = _norm_priority(PRIORITY_LABELS[int(np.argmax(pp))])

        conf_cat = float(pc.max())
        conf_pri = float(pp.max())
        ml_margin = top2_margin(pp)

        rule_pri, rule_conf, dbg = rule_priority(text, pred_category)

        RULE_ADVANTAGE = 0.15  # tune
        if rule_conf >= 0.80 and (rule_conf - conf_pri) >= RULE_ADVANTAGE:
            final_pri, final_pri_conf, pr_source = rule_pri, rule_conf, "rule"
        else:
            final_pri, final_pri_conf, pr_source = ml_priority, conf_pri, "ml"

        # Safeguard: when ML is uncertain and category base is High,
        # never let the final priority drop below Medium.
        if dbg['base'] == 'High' and conf_pri < 0.50 and final_pri == 'Low':
          final_pri = 'Medium'
          final_pri_conf = max(rule_conf, conf_pri, 0.55)
          pr_source = 'rule_safeguard'

        # Safeguard 1: base High → never output Low
        if dbg['base'] == 'High' and final_pri == 'Low':
              final_pri = 'Medium'
              final_pri_conf = max(rule_conf, conf_pri, 0.55)
              pr_source = 'rule_safeguard_base'

        # Safeguard 2: strong rule evidence (2+ med or any strong hit)
        # should override a low-margin ML decision
        if pr_source == 'ml' and rule_pri != final_pri:
            rule_evidence = dbg['hi_strong'] + dbg['hi_weak'] + dbg['med']
            if rule_evidence >= 2 and (rule_conf - conf_pri) > -0.10:
                final_pri = rule_pri
                final_pri_conf = max(rule_conf, conf_pri)
                pr_source = 'rule_evidence'

        # ✅ ALWAYS append (outside the if)
        rows.append({
            'text': text,

            # Category (ML)
            'categoryName': pred_category,
            'confidence_category': conf_cat,

            # Priority — show BOTH model + rule + final
            'ml_priority': ml_priority,
            'ml_priority_confidence': conf_pri,
            'ml_priority_margin': ml_margin,

            'rule_priority': rule_pri,
            'rule_priority_confidence': rule_conf,

            'priority': final_pri,
            'confidence_priority': final_pri_conf,
            'priority_source': pr_source,

            # Debug which patterns fired
            'priority_rule_debug': dbg,
        })

    return rows

sample_texts = [
    'I appreciate your support team, but this unresolved billing and payment mismatch is urgent for me.',
    'Claim rejected again and I need a proper review plus status update immediately.',
    'Portal crashes while changing policy details and now I cannot proceed.',
    'I cant login',
    'The quality of service is bad',
    'I have an issue with my payment online and it keeps displaying an error message before it even went into the payment page. It has been 3 days of this error i tried everyday. I would want to resolve this now because i have a payment deadline, if ever can i call to reoslve this issue',
    'Ive reported this quite a few times already and still not fixed. Please fix the login before i pay my balance which is already tomorrow.',
    'Eugene is bad',
    'not loading the billing page',
    'i was charged the wrong amount'
]

preds = predict_texts(sample_texts)
print(json.dumps(preds, indent=2))
